In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quickstart Executor

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Mirip dengan primitif [Sampler](/guides/get-started-with-sampler), Executor mengambil sampel register output dari eksekusi quantum circuit, tapi tidak memiliki error suppression atau mitigasi bawaan. Sebagai gantinya, ini adalah bagian dari [model eksekusi terarah](/guides/directed-execution-model) yang menyediakan bahan-bahan untuk menangkap niat desain di sisi klien, dan memindahkan pembuatan varian circuit yang mahal ke sisi server. Executor mengikuti direktif yang diberikan dalam anotasi dan opsi circuit, menghasilkan dan mengikat nilai parameter, mengeksekusi circuit yang terikat pada hardware, dan mengembalikan hasil eksekusi beserta metadata. Ia tidak membuat keputusan implisit apapun untukmu dan memberimu kontrol penuh serta transparansi.

> **Note:** Paket Qiskit belum memiliki kelas dasar untuk primitif Executor.

## Sebelum kamu mulai
Beberapa contoh kode di halaman ini menggunakan `samplex`, yang merupakan bagian dari paket Samplomatic. Karena itu, sebelum menjalankan blok kode tersebut, kamu harus menginstal Samplomatic, seperti yang ditunjukkan di blok kode berikut. Untuk informasi lebih lanjut, lihat [dokumentasi Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Buat dan transpile circuit
Kamu butuh setidaknya satu circuit untuk menggunakan primitif Executor. Circuit bisa secara opsional memiliki parameter.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Circuit perlu diubah agar hanya menggunakan instruksi yang didukung QPU (disebut circuit *instruction set architecture (ISA)*). Gunakan transpiler untuk melakukan ini.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inisialisasi `QuantumProgram`
Inisialisasi `QuantumProgram` dengan beban kerjamu. `QuantumProgram` terdiri dari `QuantumProgramItem`. Biasanya, setiap item terdiri dari circuit, sekumpulan nilai parameter, dan mungkin `samplex` untuk mengacak konten circuit. Untuk detail lengkap, lihat [Input dan output Executor](/guides/executor-input-output).

Sel berikut menginisialisasi `QuantumProgram` dan menentukan untuk melakukan 25 shot. Selanjutnya, ia menambahkan circuit target yang sudah ditranspile.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opsional: Kelompokkan gate dan pengukuran ke dalam kotak beranotasi
Mengelompokkan instruksi ke dalam kotak dan memberikan anotasi adalah cara utama untuk menentukan niatmu. Dalam contoh berikut, kita menggunakan `generate_boxing_pass_manager` dan parameter twirling-nya untuk mengelompokkan gate dua-qubit dan pengukuran ke dalam kotak serta menerapkan anotasi twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Jalankan Executor dan dapatkan hasil
Jalankan `QuantumProgram` pada backend IBM&reg; menggunakan primitif `Executor` dengan opsi default. Lihat [Opsi Executor](/guides/executor-options) untuk mempelajari opsi yang tersedia.